# 11.03 章节实践：独立完成位图集合交

## 小节概述

本节是一项独立实践。你将把 11.01 的位图表示和 11.02 的多核数据通路落实为可以在 NPU 上运行的集合交算子。

- <strong>建议用时：</strong> 35 分钟；
- <strong>前置要求：</strong> 已完成 [11.01 位图集合的数据结构设计](./11.01_bitmap_set_structure.ipynb) 和 [11.02 集合交算子与带宽实验](./11.02_bitmap_and_operator.ipynb)；
- <strong>修改目标：</strong> 独立工作副本中的 <code>student_compute.h</code>；
- <strong>完成标志：</strong> 源码门禁、干净构建、非法参数和 8 个 NPU 正确性用例全部通过，末尾显示 <code>SELF_CHECK PASS</code>。

> 后续单元会写入一份有意保留错误的 starter。开始修改后，不要重新运行未修改的 starter 单元，否则会覆盖已经完成的代码。

## 1. 实践任务

在不修改 Host 数据生成、行填充、计时和完整自检脚本的前提下，独立完成三个 TODO：

1. 计算当前 Block 在展平物理数组中的起始元素下标；
2. 为普通 Block 与最后一个 Block 选择正确的任务量；
3. 使用两个不同输入完成逐物理单位集合交。

验收同时覆盖 Dense/Bitmap、<code>U=31/32/33</code>、多 Block、大规模非对齐、物理填充和输出 Guard。性能数值不作为通过条件，避免设备负载造成不公平。

## 2. 准备独立工作副本

下面在 <code>/tmp/cannlab_data_structures_compute/11_bitmap_set_bandwidth/11.03_chapter_test</code> 创建独立工作副本，不修改课程的 <code>src</code> 和 <code>answer</code>。默认重复运行会保留学生代码；只有把 <code>RESET_WORKSPACE</code> 改为 <code>True</code> 时才重新复制起始工程。

In [ ]:
from pathlib import Path
import os
import shutil
import subprocess
import sys

RESET_WORKSPACE = False
WORK_ROOT = Path('/tmp/cannlab_data_structures_compute/11_bitmap_set_bandwidth/11.03_chapter_test')
current = Path.cwd().resolve()
relative = Path('contrib/tutorials/data_structures_compute/11_bitmap_set_bandwidth')

def outside_work_root(path):
    try:
        Path(path).resolve().relative_to(WORK_ROOT)
        return False
    except ValueError:
        return True

search_roots = []
previous_chapter = globals().get('CHAPTER_DIR')
if previous_chapter is not None:
    search_roots.append(Path(previous_chapter).resolve())
search_roots.extend([current, current / relative, *[p / relative for p in current.parents]])
CHAPTER_DIR = next(
    (
        candidate for candidate in search_roots
        if outside_work_root(candidate)
        and (candidate / 'src' / 'demo' / 'bitmap_and.asc').is_file()
    ),
    None,
)
if CHAPTER_DIR is None:
    raise FileNotFoundError('未找到实验 11 目录，请从 cann-learning-hub 仓库内打开 Notebook。')

previous_start = globals().get('NOTEBOOK_START')
if previous_start is not None and outside_work_root(previous_start):
    NOTEBOOK_START = Path(previous_start).resolve()
elif outside_work_root(current):
    NOTEBOOK_START = current
else:
    NOTEBOOK_START = CHAPTER_DIR

if RESET_WORKSPACE and WORK_ROOT.exists():
    os.chdir(NOTEBOOK_START)
    shutil.rmtree(WORK_ROOT)
if not WORK_ROOT.exists():
    shutil.copytree(CHAPTER_DIR / 'src', WORK_ROOT / 'src')
PROJECT = WORK_ROOT / 'src' / 'practice'
GRADER = WORK_ROOT / 'src' / 'grader' / 'grade_bitmap_and.py'
os.chdir(PROJECT)
print('project   :', PROJECT)
print('self-check:', GRADER)
print('workspace reset:', RESET_WORKSPACE)

## 3. 三个 TODO 的调用位置

<code>StudentBlockOffset</code> 只决定每核 GM 起点；<code>StudentCurrentUnits</code> 决定本核真实任务量；<code>STUDENT_COMPUTE</code> 只负责 LocalTensor 上的 AND。把三者分开能让失败信息直接对应多核、尾部或计算语义。starter 可以编译，但不会通过结果验收。

## 4. 写入 starter

修改下面单元后反复运行。不要改函数签名或宏参数。

In [ ]:
%%writefile student_compute.h
#pragma once

// TODO 1：返回当前 Block 的全局起始元素下标。
__aicore__ inline uint32_t StudentBlockOffset(uint32_t blockFormer, uint32_t blockIdx)
{
    return 0U;
}

// TODO 2：普通 Block 使用 blockFormer，最后一个 Block 使用 blockTail。
__aicore__ inline uint32_t StudentCurrentUnits(
    uint32_t blockIdx, uint32_t blockNum, uint32_t blockFormer, uint32_t blockTail)
{
    return blockTail;
}

// TODO 3：使用 x 和 y 完成集合交。
#define STUDENT_COMPUTE(z, x, y, len) \
    ComputeSetAnd((z), (x), (x), (len))

## 5. 快速自检

快速自检只检查明显占位，不代替编译与 NPU 运行。

In [ ]:
header = PROJECT / 'student_compute.h'
source = header.read_text(encoding='utf-8')
checks = {
    'TODO 已清除': 'TODO' not in source,
    '偏移不再返回 0': 'return 0U;' not in source,
    '计算使用 y': 'ComputeSetAnd((z), (x), (y)' in source,
}
for name, passed in checks.items():
    print('[PASS]' if passed else '[TODO]', name)

## 6. 运行完整自检

完整自检依次执行源码门禁、干净构建、4 个非法参数和 8 个 NPU 正确性用例。starter 会在最早的接口检查处失败；完成三个 TODO 后，末尾应显示 <code>SELF_CHECK PASS</code>。机器可读结果行不会在 Notebook 中展示。

In [ ]:
command = [sys.executable, str(GRADER), '--project', str(PROJECT), '--npu-arch', 'dav-2201']
print('$', subprocess.list2cmdline(command))
result = subprocess.run(command, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
visible_lines = []
for line in result.stdout.splitlines():
    if line.startswith('AUTO_RESULT_'):
        continue
    visible_lines.append(line.replace('[GATE PASS]', '[PASS]').replace('[GATE FAIL]', '[FAIL]'))
check_text = '\n'.join(visible_lines)
print(check_text)
print('self-check exit code:', result.returncode)
print('SELF_CHECK PASS' if result.returncode == 0 else 'SELF_CHECK FAIL')
first_failure = next((line for line in check_text.splitlines() if line.startswith('[FAIL]')), None)
if result.returncode == 0:
    print('完整自检通过。')
elif first_failure:
    print('请先处理最早的失败项：', first_failure)
else:
    print('自检未通过，请从最早的异常信息开始排查。')

## 7. 根据第一条失败信息排错

| 现象 | 优先检查 |
| --- | --- |
| 源码门禁失败 | TODO 是否清除，偏移是否同时使用两个参数，AND 是否使用 <code>x</code> 与 <code>y</code> |
| 小 Shape 通过、多 Block 失败 | <code>StudentBlockOffset</code> 是否使用 <code>blockFormer*blockIdx</code> |
| 前部正确、尾部失败 | 是否只在最后一个 Block 选择 <code>blockTail</code> |
| 所有结果像输入 A | AND 的第三个参数是否误写成 <code>x</code> |
| <code>guard=FAIL</code> | 本核任务量或 CopyOut 长度是否超过真实区间 |
| 运行时卡死或 AIC error | 先检查越界，再检查队列 <code>EnQue/DeQue</code> 是否被修改 |

## 8. 独立完成后查看参考答案

参考答案只显示，不自动覆盖工作副本。

In [ ]:
SHOW_REFERENCE_ANSWER = False
answer_dir = CHAPTER_DIR / 'answer' / '11.03_chapter_test'
if SHOW_REFERENCE_ANSWER:
    print((answer_dir / 'README.md').read_text(encoding='utf-8'))
    print((answer_dir / 'student_compute.h').read_text(encoding='utf-8'))
else:
    print('将 SHOW_REFERENCE_ANSWER 改为 True 后重新运行。')

## 9. 实践小结

完成条件如下：

1. 完整自检末尾显示 <code>SELF_CHECK PASS</code>；
2. Dense 与 Bitmap 的 <code>U=31/32/33</code> 全部通过；
3. 大规模非对齐位图和多 Block 稠密用例通过；
4. 输出 Guard 未被改写；
5. 能解释位图减少的是物理流量，而不是集合交的逻辑工作定义。

至此，你已经从集合抽象出发完成位图数据结构设计，并把表示压缩落实为可编译、可验证、可测量的 Ascend C 集合交算子。返回 [11.00 章节介绍](./11.00_chapter_intro.ipynb)。